# 02 — Gradient Descent từ đầu

Đây là **thuật toán huấn luyện của mọi mạng nơ-ron**, từ hồi quy tuyến tính đến GPT-4 / Claude / Gemini.
Chỉ khác nhau ở: mô hình to hơn, dữ liệu nhiều hơn, và gradient được tính tự động (backprop).

Vòng lặp cốt lõi:
```
lặp lại:
    1. dự đoán
    2. tính loss
    3. tính gradient của loss theo tham số
    4. tham số -= learning_rate * gradient
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)

## 1. Tìm cực tiểu của hàm một biến

$f(w) = (w - 3)^2 + 1$ có cực tiểu tại $w = 3$. Đạo hàm: $f'(w) = 2(w - 3)$.

In [ ]:
f = lambda w: (w - 3) ** 2 + 1
df = lambda w: 2 * (w - 3)

w, lr = -4.0, 0.1
history = [w]
for step in range(30):
    w = w - lr * df(w)
    history.append(w)
print(f"w sau 30 bước = {w:.4f}")

ws = np.linspace(-5, 8, 200)
plt.plot(ws, f(ws), label="f(w)")
plt.plot(history, [f(h) for h in history], "o-", ms=4, label="các bước GD")
plt.legend(); plt.xlabel("w"); plt.show()

### ✏️ Thử nghiệm
Đổi `lr` thành `0.01`, `0.5`, `0.99`, `1.1`. Chuyện gì xảy ra? Ghi nhận xét vào `notes/learning-rate.md`.
Đây chính là lý do chọn learning rate là một trong những việc quan trọng nhất khi huấn luyện LLM.

## 2. Kiểm tra đạo hàm bằng số (numerical gradient)

$$f'(w) \approx \frac{f(w+h) - f(w-h)}{2h}$$

Kỹ thuật này dùng để **kiểm tra backprop tự viết có đúng không** — bạn sẽ dùng lại ở Giai đoạn 2.

In [ ]:
def numerical_grad(f, w, h=1e-5):
    return (f(w + h) - f(w - h)) / (2 * h)

for w0 in [-1.0, 0.0, 2.5]:
    print(w0, df(w0), numerical_grad(f, w0))

## 3. Hồi quy tuyến tính bằng gradient descent

Dữ liệu giả: $y = 2x + 1 + \text{nhiễu}$. Mô hình: $\hat y = wx + b$. Loss MSE:

$$L = \frac{1}{N}\sum_i (\hat y_i - y_i)^2$$

Gradient (dùng chain rule):
$$\frac{\partial L}{\partial w} = \frac{2}{N}\sum_i (\hat y_i - y_i)\,x_i \qquad \frac{\partial L}{\partial b} = \frac{2}{N}\sum_i (\hat y_i - y_i)$$

### ✏️ Bài tập
Tự suy ra 2 công thức trên bằng tay trên giấy trước khi xem code.

In [ ]:
N = 100
X = np.random.rand(N) * 10
Y = 2 * X + 1 + np.random.randn(N)

w, b, lr = 0.0, 0.0, 0.01
losses = []
for step in range(1000):
    y_hat = w * X + b                       # 1. dự đoán
    loss = np.mean((y_hat - Y) ** 2)        # 2. loss
    dw = 2 * np.mean((y_hat - Y) * X)       # 3. gradient
    db = 2 * np.mean(y_hat - Y)
    w -= lr * dw                            # 4. cập nhật
    b -= lr * db
    losses.append(loss)

print(f"w = {w:.3f} (thật: 2), b = {b:.3f} (thật: 1), loss cuối = {losses[-1]:.3f}")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(losses); ax[0].set_yscale("log"); ax[0].set_title("Loss theo bước")
ax[1].scatter(X, Y, s=10); ax[1].plot(np.sort(X), w * np.sort(X) + b, "r"); ax[1].set_title("Đường khớp")
plt.show()

### ✏️ Bài tập nâng cao
1. Chuyển sang **mini-batch gradient descent**: mỗi bước chỉ dùng 16 mẫu ngẫu nhiên. So sánh đồ thị loss.
2. Mở rộng sang nhiều đặc trưng: `X` shape `(N, d)`, `w` shape `(d,)`, dùng `X @ w`.
3. Dùng `numerical_grad` để kiểm tra `dw`, `db` của bạn.
4. (Khó) Thử giảm số bước còn 200: `w` gần đúng nhưng `b` vẫn sai nhiều — vì sao? Gợi ý: tỉ lệ (scale) của `X`.
   Đây là lý do cần **chuẩn hóa dữ liệu** và optimizer thông minh như **Adam**.

In [ ]:
# Làm bài tập nâng cao ở đây